In [12]:
#Create workspace
import arcpy
import os

# folder for GDB
workspace_folder = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE"

# Name of new geodatabase
gdb_name = "ClassiFIRE.gdb"

# Full path
gdb_path = os.path.join(workspace_folder, gdb_name)

# Create the GDB 
if not arcpy.Exists(gdb_path):
    arcpy.management.CreateFileGDB(workspace_folder, gdb_name)

# Set it as workspace
arcpy.env.workspace = gdb_path

print("Working GDB:", gdb_path)

Working GDB: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb


In [11]:
for lyr in layers:
    print(lyr, arcpy.Exists(lyr), arcpy.Describe(lyr).dataType)

C:\Users\Melanie\Desktop\Rx_ClassiFIRE\SEFM_L_ABA_1994_2024_polys.gdb\L_BurnedArea_1994_poly True FeatureClass
C:\Users\Melanie\Desktop\Rx_ClassiFIRE\SEFM_L_ABA_1994_2024_polys.gdb\L_BurnedArea_1995_poly True FeatureClass
C:\Users\Melanie\Desktop\Rx_ClassiFIRE\SEFM_L_ABA_1994_2024_polys.gdb\L_BurnedArea_1996_poly True FeatureClass
C:\Users\Melanie\Desktop\Rx_ClassiFIRE\SEFM_L_ABA_1994_2024_polys.gdb\L_BurnedArea_1997_poly True FeatureClass
C:\Users\Melanie\Desktop\Rx_ClassiFIRE\SEFM_L_ABA_1994_2024_polys.gdb\L_BurnedArea_1998_poly True FeatureClass
C:\Users\Melanie\Desktop\Rx_ClassiFIRE\SEFM_L_ABA_1994_2024_polys.gdb\L_BurnedArea_1999_poly True FeatureClass
C:\Users\Melanie\Desktop\Rx_ClassiFIRE\SEFM_L_ABA_1994_2024_polys.gdb\L_BurnedArea_2000_poly True FeatureClass
C:\Users\Melanie\Desktop\Rx_ClassiFIRE\SEFM_L_ABA_1994_2024_polys.gdb\L_BurnedArea_2001_poly True FeatureClass
C:\Users\Melanie\Desktop\Rx_ClassiFIRE\SEFM_L_ABA_1994_2024_polys.gdb\L_BurnedArea_2002_poly True FeatureClass
C

In [13]:
#merge years 1994-2022

# Source GDB containing all the yearly layers
sefm_gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\SEFM_L_ABA_1994_2024_polys.gdb"

# Your working GDB where the merged output should go
working_gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb"

# Build list of full paths for 1994–2022
layers = [os.path.join(sefm_gdb, f"L_BurnedArea_{year}_poly")
          for year in range(1994, 2023)]

print(layers)

# Output path inside your working GDB
out_fc = os.path.join(working_gdb, "merged_94_22")

# Merge
arcpy.management.Merge(layers, out_fc)

print("Merged output saved to:", out_fc)

['C:\\Users\\Melanie\\Desktop\\Rx_ClassiFIRE\\SEFM_L_ABA_1994_2024_polys.gdb\\L_BurnedArea_1994_poly', 'C:\\Users\\Melanie\\Desktop\\Rx_ClassiFIRE\\SEFM_L_ABA_1994_2024_polys.gdb\\L_BurnedArea_1995_poly', 'C:\\Users\\Melanie\\Desktop\\Rx_ClassiFIRE\\SEFM_L_ABA_1994_2024_polys.gdb\\L_BurnedArea_1996_poly', 'C:\\Users\\Melanie\\Desktop\\Rx_ClassiFIRE\\SEFM_L_ABA_1994_2024_polys.gdb\\L_BurnedArea_1997_poly', 'C:\\Users\\Melanie\\Desktop\\Rx_ClassiFIRE\\SEFM_L_ABA_1994_2024_polys.gdb\\L_BurnedArea_1998_poly', 'C:\\Users\\Melanie\\Desktop\\Rx_ClassiFIRE\\SEFM_L_ABA_1994_2024_polys.gdb\\L_BurnedArea_1999_poly', 'C:\\Users\\Melanie\\Desktop\\Rx_ClassiFIRE\\SEFM_L_ABA_1994_2024_polys.gdb\\L_BurnedArea_2000_poly', 'C:\\Users\\Melanie\\Desktop\\Rx_ClassiFIRE\\SEFM_L_ABA_1994_2024_polys.gdb\\L_BurnedArea_2001_poly', 'C:\\Users\\Melanie\\Desktop\\Rx_ClassiFIRE\\SEFM_L_ABA_1994_2024_polys.gdb\\L_BurnedArea_2002_poly', 'C:\\Users\\Melanie\\Desktop\\Rx_ClassiFIRE\\SEFM_L_ABA_1994_2024_polys.gdb\\L_Bu

In [14]:
#% of nlcdr_11 and nlcdr_20 among detections
import arcpy

fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\merged_94_22"

# Total detections
total = int(arcpy.management.GetCount(fc)[0])

cats = ["nlcdr_11", "nlcdr_20"]
results = {}

for c in cats:
    sql = f"nlcdr_domi = '{c}'"
    lyr = "temp_layer"

    # Make a temporary layer with the filter
    arcpy.management.MakeFeatureLayer(fc, lyr, sql)

    # Count features in the filtered layer
    count = int(arcpy.management.GetCount(lyr)[0])
    pct = (count / total) * 100

    results[c] = (count, pct)

    # Clean up
    arcpy.management.Delete(lyr)

print("Total detections:", total)
for c, (count, pct) in results.items():
    print(f"{c}: {count} detections ({pct:.3f}%)")

Total detections: 2678794
nlcdr_11: 2665 detections (0.099%)
nlcdr_20: 78267 detections (2.922%)


In [15]:
#filter out detections where nlcdr_domi = 20 (developed, 2.9% of detections) or 11 (open water, 0.1%) 
import arcpy
import os

in_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\merged_94_22"
out_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\merged_94_22_nlcd_filtered"

# SQL expression to EXCLUDE the two categories
sql = "nlcdr_domi NOT IN ('nlcdr_11', 'nlcdr_20')"

# Export the filtered subset
arcpy.management.MakeFeatureLayer(in_fc, "temp_lyr", sql)
arcpy.management.CopyFeatures("temp_lyr", out_fc)
arcpy.management.Delete("temp_lyr")

print("Filtered output saved to:", out_fc)

Filtered output saved to: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\merged_94_22_nlcd_filtered


In [16]:
#determine % of detections below 0.81 ha (2 acres) (mimimum detectable size reported by SEFM is 0.89 ha) (0% only two detections excluded)
import arcpy
import os

# Input (already NLCD-filtered)
in_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\merged_94_22_nlcd_filtered"

# Output (size-filtered)
out_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\merged_94_22_nlcd_size_filtered"

# -------------------------------------------------------------------
# 1. Add area_ha field
# -------------------------------------------------------------------
fields = [f.name for f in arcpy.ListFields(in_fc)]
if "area_ha" not in fields:
    arcpy.management.AddField(in_fc, "area_ha", "DOUBLE")

# -------------------------------------------------------------------
# 2. Calculate area in hectares
# -------------------------------------------------------------------
arcpy.management.CalculateField(
    in_fc,
    "area_ha",
    "!shape.area@SQUAREMETERS! / 10000",
    "PYTHON3"
)

# -------------------------------------------------------------------
# 3. Calculate % of detections < 0.81 ha
# -------------------------------------------------------------------
# Total detections
total = int(arcpy.management.GetCount(in_fc)[0])

# Make a temporary layer for small detections
small_sql = "area_ha < 0.81"
arcpy.management.MakeFeatureLayer(in_fc, "small_lyr", small_sql)
small_count = int(arcpy.management.GetCount("small_lyr")[0])

pct_small = (small_count / total) * 100

print(f"Total detections: {total}")
print(f"Detections < 0.81 ha: {small_count} ({pct_small:.3f}%)")

# Clean up
arcpy.management.Delete("small_lyr")

# -------------------------------------------------------------------
# 4. Export detections >= 0.81 ha
# -------------------------------------------------------------------
keep_sql = "area_ha >= 0.81"

arcpy.management.MakeFeatureLayer(in_fc, "keep_lyr", keep_sql)
arcpy.management.CopyFeatures("keep_lyr", out_fc)
arcpy.management.Delete("keep_lyr")

print("Filtered output saved to:", out_fc)

Total detections: 2597862
Detections < 0.81 ha: 2 (0.000%)
Filtered output saved to: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\merged_94_22_nlcd_size_filtered


In [17]:
#assign each remaining detection a unique "detection_id" for tracking
import arcpy

fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\merged_94_22_nlcd_size_filtered"

# 1. Add numeric detection_id field
fields = [f.name for f in arcpy.ListFields(fc)]
if "detection_id" not in fields:
    arcpy.management.AddField(fc, "detection_id", "LONG")

# 2. Populate detection_id with sequential integers
with arcpy.da.UpdateCursor(fc, ["detection_id"]) as cursor:
    counter = 1
    for row in cursor:
        row[0] = counter
        cursor.updateRow(row)
        counter += 1

print("Numeric detection_id assigned.")

Numeric detection_id assigned.


In [18]:
#we want to use prebd_min for temporal comparisons, but some values are invalid. Why and how many?
# determine how many prebd_min values are invalid
import arcpy

detections = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\merged_94_22_nlcd_size_filtered"

total = 0
valid = 0

zero_vals = 0
invalid_month = 0
invalid_day = 0
malformed = 0

with arcpy.da.SearchCursor(detections, ["prebd_min"]) as cur:
    for (prebd,) in cur:
        total += 1

        # Case 1: zero or null
        if prebd in (0, None):
            zero_vals += 1
            continue

        # Convert float → int safely
        try:
            val = int(prebd)
        except Exception:
            malformed += 1
            continue

        s = str(val)

        # Must be exactly 8 digits (YYYYMMDD)
        if len(s) != 8:
            malformed += 1
            continue

        year = int(s[0:4])
        month = int(s[4:6])
        day = int(s[6:8])

        # Validate month/day ranges
        if not (1 <= month <= 12):
            invalid_month += 1
            continue

        if not (1 <= day <= 31):
            invalid_day += 1
            continue

        # If all checks passed
        valid += 1

# Summary
print("Total detections:", total)
print("Valid prebd_min:", valid)
print("Invalid total:", total - valid)

print("\nBreakdown of invalid values:")
print("  Zero values:", zero_vals)
print("  Invalid month:", invalid_month)
print("  Invalid day:", invalid_day)
print("  Malformed:", malformed)

def pct(x):
    return round((x / total) * 100, 2)

print("\nPercentages:")
print("  Zero values:", pct(zero_vals), "%")
print("  Invalid month:", pct(invalid_month), "%")
print("  Invalid day:", pct(invalid_day), "%")
print("  Malformed:", pct(malformed), "%")
print("  Valid:", pct(valid), "%")

Total detections: 2597860
Valid prebd_min: 2467527
Invalid total: 130333

Breakdown of invalid values:
  Zero values: 5666
  Invalid month: 0
  Invalid day: 124667
  Malformed: 0

Percentages:
  Zero values: 0.22 %
  Invalid month: 0.0 %
  Invalid day: 4.8 %
  Malformed: 0.0 %
  Valid: 94.98 %


In [19]:
#For prebd_min = 0, set those aside. I will deal with them later. Exclude from other data. 
import arcpy

arcpy.env.overwriteOutput = True

detections = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\merged_94_22_nlcd_size_filtered"
out_gdb = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb"

zero_fc = f"{out_gdb}\\merged_94_22_nlcd_size_filtered_prebd_min_0"

arcpy.management.MakeFeatureLayer(detections, "det_lyr_zero")
arcpy.management.SelectLayerByAttribute(
    "det_lyr_zero",
    "NEW_SELECTION",
    "prebd_min = 0"
)
arcpy.management.CopyFeatures("det_lyr_zero", zero_fc)

print("Created:", zero_fc)

nonzero_fc = f"{out_gdb}\\merged_94_22_nlcd_size_filtered_prebd_min_nonzero"

arcpy.management.MakeFeatureLayer(detections, "det_lyr_nonzero")
arcpy.management.SelectLayerByAttribute(
    "det_lyr_nonzero",
    "NEW_SELECTION",
    "prebd_min <> 0"
)
arcpy.management.CopyFeatures("det_lyr_nonzero", nonzero_fc)

print("Created:", nonzero_fc)

Created: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\merged_94_22_nlcd_size_filtered_prebd_min_0
Created: C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\merged_94_22_nlcd_size_filtered_prebd_min_nonzero


In [20]:
#Using those data where prebd_min is not equal to zero, we need to create a column where prebd_min day can be corrected. 
import arcpy

arcpy.env.overwriteOutput = True

nonzero_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\merged_94_22_nlcd_size_filtered_prebd_min_nonzero"

# Add corrected field if needed
fields = [f.name for f in arcpy.ListFields(nonzero_fc)]
if "prebd_min_corrected" not in fields:
    arcpy.management.AddField(nonzero_fc, "prebd_min_corrected", "LONG")

with arcpy.da.UpdateCursor(nonzero_fc, ["prebd_min", "prebd_min_corrected"]) as cur:
    for row in cur:
        prebd = row[0]

        # Convert float → int
        val = int(prebd)
        s = str(val)

        # Extract components
        year = int(s[0:4])
        month = int(s[4:6])
        day = int(s[6:8])

        # Clamp day into 1–31
        if day < 1:
            day = 1
        elif day > 31:
            day = 31

        # Rebuild corrected YYYYMMDD
        fixed = int(f"{year:04d}{month:02d}{day:02d}")

        # Write corrected value
        row[1] = fixed
        cur.updateRow(row)

print("Corrected prebd_min values written to prebd_min_corrected.")

Corrected prebd_min values written to prebd_min_corrected.


In [22]:
#we want to use bd_min for temporal comparisons, but some values are invalid. Why and how many?
# check validity of bd_min values
import arcpy

detections = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\merged_94_22_nlcd_size_filtered"

total = 0
valid = 0

zero_vals = 0
invalid_month = 0
invalid_day = 0
malformed = 0

with arcpy.da.SearchCursor(detections, ["bd_min"]) as cur:
    for (bd,) in cur:
        total += 1

        # Case 1: zero or null
        if bd in (0, None):
            zero_vals += 1
            continue

        # Convert float → int
        try:
            val = int(bd)
        except Exception:
            malformed += 1
            continue

        s = str(val)

        # Must be 8 digits
        if len(s) != 8:
            malformed += 1
            continue

        year = int(s[0:4])
        month = int(s[4:6])
        day = int(s[6:8])

        # Check month/day
        if not (1 <= month <= 12):
            invalid_month += 1
            continue

        if not (1 <= day <= 31):
            invalid_day += 1
            continue

        valid += 1

# Summary
print("Total detections:", total)
print("Valid bd_min:", valid)
print("Invalid total:", total - valid)

print("\nBreakdown of invalid values:")
print("  Zero values:", zero_vals)
print("  Invalid month:", invalid_month)
print("  Invalid day:", invalid_day)
print("  Malformed:", malformed)

def pct(x):
    return round((x / total) * 100, 2)

print("\nPercentages:")
print("  Zero values:", pct(zero_vals), "%")
print("  Invalid month:", pct(invalid_month), "%")
print("  Invalid day:", pct(invalid_day), "%")
print("  Malformed:", pct(malformed), "%")
print("  Valid:", pct(valid), "%")

Total detections: 2597860
Valid bd_min: 2465680
Invalid total: 132180

Breakdown of invalid values:
  Zero values: 5666
  Invalid month: 0
  Invalid day: 126514
  Malformed: 0

Percentages:
  Zero values: 0.22 %
  Invalid month: 0.0 %
  Invalid day: 4.87 %
  Malformed: 0.0 %
  Valid: 94.91 %


In [23]:
#Ok prebd_min is corrected. Now I need to correct bd_min: 
import arcpy

nonzero_fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\merged_94_22_nlcd_size_filtered_prebd_min_nonzero"

# Add corrected field if needed
fields = [f.name for f in arcpy.ListFields(nonzero_fc)]
if "bd_min_corrected" not in fields:
    arcpy.management.AddField(nonzero_fc, "bd_min_corrected", "LONG")

with arcpy.da.UpdateCursor(nonzero_fc, ["bd_min", "bd_min_corrected"]) as cur:
    for row in cur:
        bdmin = row[0]

        # Convert float → int
        val = int(bdmin)
        s = str(val)

        # Extract components
        year = int(s[0:4])
        month = int(s[4:6])
        day = int(s[6:8])

        # Clamp day into 1–31
        if day < 1:
            day = 1
        elif day > 31:
            day = 31

        # Rebuild corrected YYYYMMDD
        fixed = int(f"{year:04d}{month:02d}{day:02d}")

        # Write corrected value
        row[1] = fixed
        cur.updateRow(row)

print("Corrected bd_min values written to bd_min_corrected.")

Corrected bd_min values written to bd_min_corrected.


In [24]:
#Add 8 days to bd_min_corrected for end of time interval: 
import arcpy
from datetime import datetime, timedelta

fc = r"C:\Users\Melanie\Desktop\Rx_ClassiFIRE\ClassiFIRE.gdb\merged_94_22_nlcd_size_filtered_prebd_min_nonzero"

# Add field if missing
fields = [f.name for f in arcpy.ListFields(fc)]
if "bd_min_corrected_plus8" not in fields:
    arcpy.management.AddField(fc, "bd_min_corrected_plus8", "LONG")

with arcpy.da.UpdateCursor(fc, ["bd_min_corrected", "bd_min_corrected_plus8"]) as cur:
    for row in cur:
        bdmin_corr = row[0]

        # Convert YYYYMMDD → datetime
        s = str(int(bdmin_corr))
        year  = int(s[0:4])
        month = int(s[4:6])
        day   = int(s[6:8])

        date_obj = datetime(year, month, day)

        # Add 8 days
        new_date = date_obj + timedelta(days=8)

        # Convert back to YYYYMMDD integer
        row[1] = int(new_date.strftime("%Y%m%d"))

        cur.updateRow(row)

print("bd_min_corrected_plus8 field populated.")

bd_min_corrected_plus8 field populated.
